# Inspect an embedding Codenames game

The Cursor terminal often shows Hebrew backwards. Open a JSON log here instead.

Logs live in `results/embedding/` (gitignored). Older `embeddings_method` files may still be under `results/`.

The last section prints a recap of each game by type (same-model Word2Vec, same-model fastText, later cross/concat). Default: seeds `0–4` on the intersection regular pool.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "codenames" / "board.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULTS = ROOT / "results" / "embedding"
print("repo root:", ROOT)
print("logs:", RESULTS)

repo root: /Users/sharons/personal-projects/nlp_project_reichman
logs: /Users/sharons/personal-projects/nlp_project_reichman/results/embedding


In [2]:
def latest_embedding_log() -> Path:
    files = sorted(
        RESULTS.glob("single_team_*_embedding_embedding_*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not files:
        raise FileNotFoundError(f"No embedding game JSON in {RESULTS}")
    return files[0]

path = latest_embedding_log()
game = json.loads(path.read_text(encoding="utf-8"))
print(path.name)
print(
    f"seed={game['seed']}  wordpool={game['wordpool']}  "
    f"outcome={game['outcome']}  turns={game['num_turns']}  score={game['score']}"
)
print(f"model={game.get('model')}  params={game.get('model_params')}")

single_team_seed0_regular_in_vocab_intersection_fasttext_word2vec_embedding_embedding_20260827T181210130538Z.json
seed=0  wordpool=regular_in_vocab_intersection_fasttext_word2vec  outcome=win  turns=8  score=8
model=fasttext  params={'threshold': 0.4, 'candidate_limit': 20000, 'codemaster_model': 'fasttext', 'guesser_model': 'fasttext', 'clues_restricted_to_guesser_vocab': False}


In [3]:
roles = game["key_grid"]
words = game["board_words"]
board_df = pd.DataFrame(
    {"word": words, "role": roles}
)
display(board_df.groupby("role")["word"].apply(list).to_frame())

,word
role,
Assassin,[חלל]
Blue,"[גנב, אייל, כדור, כינור, יחידה, טעם, כוס, להקה]"
Civilian,"[זוהר, ספינה, פרק, קברן, טייס, סגור, קיר]"
Red,"[סיכה, עוקץ, דינוזאור, לילה, נקודה, מוות, פירא..."


In [4]:
rows = []
for turn in game["turns"]:
    guesses = ", ".join(
        f"{g['word']} ({g['role']})" for g in turn["guesses"]
    )
    rows.append(
        {
            "turn": turn["turn"],
            "clue": turn["clue"],
            "n": turn["clue_num"],
            "targets": ", ".join(turn.get("parsed_targets") or []),
            "min_target": turn.get("min_target"),
            "max_bad": turn.get("max_bad"),
            "guesses": guesses,
        }
    )
display(pd.DataFrame(rows))

,turn,clue,n,targets,min_target,max_bad,guesses
0,1,עיגול,2,"נקודה, מעגל",0.408575,0.360110,"מעגל (Red), נקודה (Red)"
1,2,עונש,1,מוות,0.601513,0.242282,מוות (Red)
2,3,המאובנים,1,דינוזאור,0.567611,0.213602,דינוזאור (Red)
3,4,הערב,1,לילה,0.541905,0.323070,לילה (Red)
4,5,פלסטיק,1,סיכה,0.505675,0.316634,סיכה (Red)
5,6,נטול,1,בעל,0.489535,0.354195,בעל (Red)
6,7,דבורים,1,עוקץ,0.439337,0.251447,עוקץ (Red)
7,8,קפטן,1,פיראט,0.435701,0.349934,פיראט (Red)


## Recaps by type (threshold grid)

Summary across **0.2 / 0.4 / 0.6**, then clue tables per model and threshold. Seeds 0–4, intersection regular. A missing seed at 0.6 means **no safe clue** (the runner crashed; no JSON). Re-run after new games.

In [10]:
import importlib

import codenames.summarize_embedding as summarize_embedding

importlib.reload(summarize_embedding)
from IPython.display import HTML, Markdown

from codenames.summarize_embedding import (
    load_embedding_games,
    overview_rows,
    select_games,
    turn_rows,
)

SEEDS = [0, 1, 2, 3, 4]
WORDPOOL = "regular_in_vocab_intersection_fasttext_word2vec"
THRESHOLDS = [0.2, 0.4, 0.6]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)


def show_table(df: pd.DataFrame) -> None:
    html = df.to_html(index=False, escape=True)
    display(HTML(f'<div style="max-height:none;overflow:visible">{html}</div>'))


selected = select_games(
    load_embedding_games(RESULTS),
    seeds=SEEDS,
    wordpool=WORDPOOL,
)
if not selected:
    print("No matching games yet.")
else:
    overview = pd.DataFrame(overview_rows(selected))
    overview = overview.sort_values(["model", "threshold", "seed"])
    display(Markdown("### Games (logged only; missing 0.6 rows = no safe clue)"))
    show_table(overview)
    display(Markdown("### Mean turns by model and threshold"))
    show_table(
        overview.groupby(["model", "threshold"], as_index=False).agg(
            n=("seed", "count"),
            wins=("outcome", lambda s: int((s == "win").sum())),
            mean_turns=("turns", "mean"),
            mean_wrong=("wrong", "mean"),
        )
    )
    for model_name in sorted({str(g.get("model")) for g in selected}):
        for thr in THRESHOLDS:
            batch = [
                g
                for g in selected
                if g.get("model") == model_name
                and (g.get("model_params") or {}).get("threshold") == thr
            ]
            display(Markdown(f"### {model_name}  threshold={thr}  ({len(batch)}/5 games logged)"))
            if not batch:
                display(Markdown("_No logs: no safe clue on these seeds._"))
                continue
            show_table(pd.DataFrame(turn_rows(batch)))

### Games (logged only; missing 0.6 rows = no safe clue)

model,seed,threshold,outcome,turns,wrong,assassin,clue_ns
fasttext,0,0.2,win,2,0,False,"6,3"
fasttext,1,0.2,win,3,0,False,"5,3,1"
fasttext,2,0.2,win,2,0,False,"6,3"
fasttext,3,0.2,win,3,0,False,"5,3,1"
fasttext,4,0.2,win,3,0,False,"4,3,2"
fasttext,0,0.4,win,8,0,False,"2,1,1,1,1,1,1,1"
fasttext,1,0.4,win,8,0,False,"2,1,1,1,1,1,1,1"
fasttext,2,0.4,win,8,0,False,"2,1,1,1,1,1,1,1"
fasttext,3,0.4,win,9,0,False,"1,1,1,1,1,1,1,1,1"
fasttext,4,0.4,win,7,0,False,"2,2,1,1,1,1,1"


### Mean turns by model and threshold

model,threshold,n,wins,mean_turns,mean_wrong
fasttext,0.2,5,5,2.600000,0.0
fasttext,0.4,5,5,8.000000,0.0
word2vec,0.2,5,5,2.200000,0.0
word2vec,0.4,5,5,3.600000,0.0
word2vec,0.6,3,3,8.666667,0.0


### fasttext  threshold=0.2  (5/5 games logged)

...

seed,turn,clue,n,targets,guesses,min_t,max_bad
0,1,כואב,6,"סיכה, עוקץ, לילה, נקודה, מוות, מעגל","עוקץ (Red), מוות (Red), סיכה (Red), נקודה (Red), לילה (Red), מעגל (Red)",0.341,0.335
0,2,זן,3,"דינוזאור, פיראט, בעל","דינוזאור (Red), בעל (Red), פיראט (Red)",0.343,0.326
1,1,סיירוס,6,"תמנון, אתר, פינגווין, גל, נסיכה, מכשפה","פינגווין (Red), מכשפה (Red), אתר (Red), נסיכה (Red), גל (Red), תמנון (Red)",0.316,0.297
1,2,מעמד,3,"מסע, חבר, זהב","זהב (Red), חבר (Red), מסע (Red)",0.319,0.303
2,1,הריאות,5,"אוויר, טבע, חיבור, זכוכית, בשר","אוויר (Red), בשר (Red), חיבור (Red), טבע (Red), זכוכית (Red)",0.318,0.312
2,2,שוחד,4,"רולטה, זוהר, זהב, ראש","זהב (Red), ראש (Red), רולטה (Red), זוהר (Red)",0.258,0.256
3,1,להזכיר,7,"מופע, טעם, לשון, ארץ, חוליה, רגל, נתיב","נתיב (Red), ארץ (Red), רגל (Red), חוליה (Red), טעם (Red), מופע (Red), לשון (Red)",0.266,0.262
3,2,ג'ק,2,"דרקון, ברמודה","ברמודה (Red), דרקון (Red)",0.514,0.500
4,1,פוריות,5,"תת, מצח, ארץ, מים, מלחמה","מים (Red), תת (Red), מלחמה (Red), ארץ (Red), מצח (Red)",0.320,0.306
4,2,לורן,3,"כותנה, טבעת, סוכן","טבעת (Red), כותנה (Red), סוכן (Red)",0.342,0.308


### word2vec  threshold=0.4  (5/5 games logged)

seed,turn,clue,n,targets,guesses,min_t,max_bad
0,1,נוצות,4,"סיכה, עוקץ, דינוזאור, פיראט","עוקץ (Red), סיכה (Red), דינוזאור (Red), פיראט (Red)",0.424,0.366
0,2,המוגדר,3,"נקודה, בעל, מעגל","מעגל (Red), נקודה (Red), בעל (Red)",0.448,0.396
0,3,מתעורר,2,"לילה, מוות","מוות (Red), לילה (Red)",0.505,0.498
1,1,הלו,4,"תמנון, פינגווין, נסיכה, מכשפה","מכשפה (Red), פינגווין (Red), נסיכה (Red), תמנון (Red)",0.490,0.445
1,2,שליחות,2,"מסע, זהב","זהב (Red), מסע (Red)",0.470,0.383
1,3,קישור,2,"אתר, חבר","אתר (Red), חבר (Red)",0.434,0.370
1,4,רדיוס,1,גל,גל (Red),0.632,0.456
2,1,מיקרוגל,4,"אוויר, חיבור, זכוכית, בשר","חיבור (Red), זכוכית (Red), אוויר (Red), בשר (Red)",0.424,0.423
2,2,ספיר,3,"זוהר, זהב, ראש","זוהר (Red), זהב (Red), ראש (Red)",0.409,0.371
2,3,שעשועים,2,"רולטה, טבע","טבע (Red), רולטה (Red)",0.447,0.418


### word2vec  threshold=0.6  (3/5 games logged)

seed,turn,clue,n,targets,guesses,min_t,max_bad
2,1,מבושל,1,בשר,בשר (Red),0.858,0.639
2,2,האויר,1,אוויר,אוויר (Red),0.818,0.422
2,3,פורצלן,1,זכוכית,זכוכית (Red),0.801,0.524
2,4,והגנים,1,טבע,טבע (Red),0.773,0.342
2,5,לאינטרנט,1,חיבור,חיבור (Red),0.769,0.474
2,6,מאור,1,זוהר,זוהר (Red),0.742,0.359
2,7,יושב,1,ראש,ראש (Red),0.712,0.404
2,8,מדליות,1,זהב,זהב (Red),0.687,0.333
2,9,פוקר,1,רולטה,רולטה (Red),0.671,0.401
3,1,הכתף,2,"חוליה, רגל","חוליה (Red), רגל (Red)",0.628,0.470
